# Person 4 (Neha): English-only check of the Hinglish detector
**Exploratory. Not in the frozen analysis plan.** Question: does the Hinglish detector wrongly flag ordinary English prompts? If it never fires on English, a Hinglish-only correction never changes English routing.

Provenance note: `models/hinglish_language_detector.joblib` was made earlier and its training data is not recorded. So this notebook draws **fresh** English prompts (fixed seed) and measures false positives on them. Recall is measured on the 32 outside Hinglish prompts that Neha wrote after the detector was trained.

Run the cells top to bottom, once.

In [ ]:
!pip -q install datasets
import hashlib, json, math, os, platform, datetime
import numpy as np, pandas as pd, joblib, sklearn
from google.colab import files
os.makedirs('/content/out', exist_ok=True)
print('sklearn', sklearn.__version__)

**Step 1.** Upload two files from your repo folder: `models/hinglish_language_detector.joblib` and `results/outside_hinglish_prompt_audit_v1.csv`.

In [ ]:
up = files.upload()
DET = 'hinglish_language_detector.joblib'
OUT = 'outside_hinglish_prompt_audit_v1.csv'
assert DET in up and OUT in up, 'Upload both files (names must match exactly)'
det = joblib.load(DET)
det_hash = hashlib.sha256(open(DET,'rb').read()).hexdigest()
out_hash = hashlib.sha256(open(OUT,'rb').read()).hexdigest()
classes = list(det.classes_); print(classes)
HI = classes.index('Hinglish')
print('detector sha256', det_hash)

**Step 2.** Draw fresh English prompts. Seed 0, fixed sizes. If a dataset name fails, paste the error back to me.

In [ ]:
from datasets import load_dataset
SEED = 0
sets = {}

d = load_dataset('cais/mmlu', 'all', split='test').to_pandas()
d = d.sample(200, random_state=SEED)
sets['MMLU'] = pd.DataFrame({'item_id': d.index.astype(str),
    'text': [q + '\n' + '\n'.join(f'{l}. {c}' for l, c in zip('ABCD', ch)) for q, ch in zip(d['question'], d['choices'])]})

d = load_dataset('openai/gsm8k', 'main', split='test').to_pandas()
d = d.sample(200, random_state=SEED)
sets['GSM8K'] = pd.DataFrame({'item_id': d.index.astype(str), 'text': d['question'].values})

d = load_dataset('HuggingFaceH4/mt_bench_prompts', split='train').to_pandas()
sets['MT-Bench'] = pd.DataFrame({'item_id': d['prompt_id'].astype(str).values, 'text': [p[0] for p in d['prompt']]})

for k, v in sets.items(): print(k, len(v))

**Step 3.** False positives: English prompts the detector calls Hinglish. Wilson 95% intervals.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; dd = 1 + z*z/n; c = (p + z*z/(2*n)) / dd
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / dd
    return round(max(0, c-h), 4), round(min(1, c+h), 4)

rows, detail = [], []
for name, df in sets.items():
    prob = det.predict_proba(df['text'].tolist())[:, HI]
    flag = prob >= 0.5
    rows.append({'dataset': name, 'n': len(df), 'flagged_hinglish': int(flag.sum()),
                 'false_positive_rate': flag.mean(), 'ci95_low': wilson(int(flag.sum()), len(df))[0],
                 'ci95_high': wilson(int(flag.sum()), len(df))[1]})
    detail.append(pd.DataFrame({'dataset': name, 'item_id': df['item_id'], 'p_hinglish': prob, 'flagged': flag}))
tot_k = sum(r['flagged_hinglish'] for r in rows); tot_n = sum(r['n'] for r in rows)
rows.append({'dataset': 'ALL', 'n': tot_n, 'flagged_hinglish': tot_k, 'false_positive_rate': tot_k/tot_n,
             'ci95_low': wilson(tot_k, tot_n)[0], 'ci95_high': wilson(tot_k, tot_n)[1]})
fp = pd.DataFrame(rows); print(fp.to_string(index=False))
pd.concat(detail).to_csv('/content/out/english_only_items_v1.csv', index=False)

**Step 4.** Recall: the 32 outside Hinglish prompts (written by Neha after the detector was trained).

In [ ]:
o = pd.read_csv(OUT)
p = det.predict_proba(o['prompt'].tolist())[:, HI]
hit = p >= 0.5
rec = pd.DataFrame([{'set': 'outside_hinglish_32', 'n': len(o), 'detected': int(hit.sum()),
                     'recall': hit.mean(), 'ci95_low': wilson(int(hit.sum()), len(o))[0],
                     'ci95_high': wilson(int(hit.sum()), len(o))[1]}])
print(rec.to_string(index=False))
print('Missed prompts:'); print(o.loc[~hit, ['prompt_id', 'prompt']].to_string(index=False))

**Step 5.** Save the results and a config (versions, seed, hashes). Then download the zip.

In [ ]:
fp.to_csv('/content/out/english_only_summary_v1.csv', index=False)
rec.to_csv('/content/out/detector_recall_v1.csv', index=False)
cfg = {'date_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z', 'seed': SEED,
       'sample_sizes': {'MMLU': 200, 'GSM8K': 200, 'MT-Bench': 'all 80 first-turn prompts'},
       'datasets': ['cais/mmlu all/test', 'openai/gsm8k main/test', 'HuggingFaceH4/mt_bench_prompts train'],
       'detector_file': 'models/hinglish_language_detector.joblib', 'detector_sha256': det_hash,
       'detector_training_data': 'NOT RECORDED (legacy file)', 'outside_prompts_sha256': out_hash,
       'threshold': 0.5, 'sklearn': sklearn.__version__, 'python': platform.python_version(),
       'status': 'exploratory, not in the frozen plan'}
json.dump(cfg, open('/content/out/config_english_only_v1.json', 'w'), indent=2)
!cd /content/out && zip -q english_only_check_v1.zip *.csv *.json && ls -la
files.download('/content/out/english_only_check_v1.zip')